# main

In [ ]:
#| export
import base64,csv,io,os,random,secrets
from datetime import date
import matplotlib,matplotlib.pyplot as plt
from fasthtml.common import *
from fasthtml.jupyter import *
from fastlite import *
from monsterui.all import *

matplotlib.use('Agg')

In [ ]:
#| export
login_redir = Redirect('/login')

def before(req, sess):
    auth = req.scope['auth'] = sess.get('auth', None)
    if not auth: return login_redir

bware = Beforeware(before, skip=[r'/favicon\.ico', '/login', '/'])
app,rt = fast_app(before=bware, session_cookie='cobalt_sess', secret_key=secrets.token_hex(32), hdrs=Theme.blue.headers(katex=True))

In [ ]:
def get_preview(app): return partial(HTMX, app=app, host=None, port=None)
p = get_preview(app)

In [ ]:
if 'srv' not in globals(): srv = JupyUvi(app=app)

In [ ]:
#| export
def mk_state(): return dict(reversals=[], trial=0, gap_ms=100, results=[], correct_streak=0, last_direction=None)

In [ ]:
#| export
def get_state(session):
    if 'state' not in session: session['state'] = mk_state()
    return session['state']

In [ ]:
#| export
@rt('/favicon.ico')
def favicon(): return FileResponse('favicon.ico')

In [ ]:
#| export
@rt
def stimulus(): return FileResponse('stimulus.wav')

In [ ]:
#| export
RESEARCHER_USER,RESEARCHER_PASS = os.getenv('RESEARCHER_USER'),os.getenv('RESEARCHER_PASS')

Relevant documentation for layout changes:

1. **MonsterUI docs** — the MonsterUI library is built on top of DaisyUI/Tailwind, so the [MonsterUI docs](https://monsterui.answer.ai) are the best starting point for components like `ButtonT`, `Card`, `DivFullySpaced`, etc.

2. **Tailwind CSS utility classes** — for layout classes like `min-h-screen`, `flex`, `items-center`, `justify-center`, `gap-4`, `mt-4`, `w-96` — these come straight from [Tailwind's docs](https://tailwindcss.com/docs). Search for the property you want (e.g. "justify content") and it'll show you the utility class name.

In general: if it looks like a component (e.g. `ButtonT.primary`, `Card`, `DivFullySpaced`) → MonsterUI; if it's a plain string in `cls=` → Tailwind.

In [ ]:
#| export
def mk_nav(auth=None):
    authed = [A('Test', href=test),
              A('Data', href=data),
              A('Logout', href=logout)]
    anon   = [A('Login',  href='/login')]
    return NavBar(*(authed if auth else anon),
                  brand=A(H3('COBALT'), href=index),
                  id='navbar', cls='bg-primary text-primary-content py-4 px-16')

In [ ]:
#| export
def mk_shell(sess, content): return Title('COBALT'), mk_nav(sess.get('auth', None)), Main(content, id='main')

## Auth

In [ ]:
#| export
@rt('/login')
def get(sess):
    if sess.get('auth'): return Redirect('/')
    return mk_shell(sess, Div(Card(
        H3('Researcher Login'),
        P(id='errormsg', cls='text-red-500'),
        Form(LabelInput('Username', id='username', name='username', autocomplete='username'),
             LabelInput('Password', id='password', name='password', type='password', autocomplete='current-password'),
             Button('Log in', cls=ButtonT.primary),
             hx_post='/login', hx_target='#errormsg'), cls='w-96'), cls='flex justify-center pt-4'))

@rt('/login')
def post(req, sess, username:str, password:str):
    if username==RESEARCHER_USER and password==RESEARCHER_PASS: sess['auth'] = True
    if not sess.get('auth'): return 'Wrong username or password.'
    return Redirect('/')

@rt
def logout(req, sess):
    sess.clear()
    return Redirect('/')

How do I protect my login route from brute-force attacks?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Great question! There are a few common strategies, and they layer nicely:

1. **Rate limiting** — track failed attempts per IP (or username) and reject/delay after a threshold (e.g. 5 failures in 60 seconds)
2. **Exponential backoff** — increase the delay between allowed attempts after each failure
3. **Account lockout** — temporarily lock the account after N failures
4. **CAPTCHA** — add a challenge after a few failures (heavier to implement)

For your setup, the simplest starting point is **in-memory rate limiting** in your `POST /login` handler — keep a dict mapping IPs to `(fail_count, last_fail_time)`, and return an error if they've exceeded the threshold within a time window.

Want to start with that approach? I can walk you through the key pieces.

Best practices for making a reproducible shell site-wide? I was thinking of turning mk_shell into a decorator, but maybe there's a better way?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠



<details class='tool-usage-details'>
<summary>read_url(url='https://docs.fastht.ml/explains/routes.html')</summary>

```json
{
  "id": "toolu_01YV2dr14GPw8ki6WfYSjMSk",
  "call": {
    "function": "read_url",
    "arguments": {
      "url": "https://docs.fastht.ml/explains/routes.html"
    }
  },
  "result": "<TRUNCATED>\u2026-----------------------------------------------------------------\ngaierror                                  Traceback (most recent call last)\nFile /usr/local/lib/python3.12/site-packages/urllib3/connection.py:204, in HTTPConnection._new_conn(self)\n    203 try:\n--> 204     sock = connection.create_connection(\n    205         (self._dns_host, self.port),\n    206         self.timeout,\n    207         source_address=self.source_address,\n    208         socket_options=self.socket_options,\n    209     )\n    210 except socket.gaierror as e:\n\nFile /usr/local/lib/python3.12/site-packages/urllib3/util/connection.py:60, in create_connection(address, timeout, source_address, socket_options)\n     58     raise LocationParseError(f\"'{host}', label empty or too long\") from None\n---> 60 for res in socket.getaddrinfo(host, port, family, socket.SOCK_STREAM):\n     61     af, socktype, proto, canonname, sa = res\n\nFile /usr/local/lib/python3.12/socket.py:978, in getaddrinfo(host, port, f\u2026</TRUNCATED>"
}
```

</details>



🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠



<details class='tool-usage-details'>
<summary>read_url(url='<…>…w.githubusercontent.com/Answer…</…>', selector='layout')</summary>

```json
{
  "id": "toolu_01VMGvi5EkbBUmtMEeXfw1ZP",
  "call": {
    "function": "read_url",
    "arguments": {
      "url": "https://raw.githubusercontent.com/AnswerDotAI/fasthtml/main/fasthtml/core.py",
      "selector": "layout"
    }
  },
  "result": "<TRUNCATED>\u2026stHTML` subclass of `Starlette`.\"\"\"\n\n# AUTOGENERATED! DO NOT EDIT! File to edit: ../nbs/api/00_core.ipynb.\n\n# %% auto #0\n__all__ = ['empty', 'htmx_hdrs', 'fh_cfg', 'htmx_resps', 'htmx_exts', 'htmxsrc', 'fhjsscr', 'surrsrc', 'scopesrc', 'viewport',\n           'charset', 'cors_allow', 'iframe_scr', 'all_meths', 'devtools_loc', 'parsed_date', 'snake2hyphens',\n           'HtmxHeaders', 'HttpHeader', 'HtmxResponseHeaders', 'form2dict', 'parse_form', 'ApiReturn', 'JSONResponse',\n           'flat_xt', 'Beforeware', 'EventStream', 'signal_shutdown', 'uri', 'decode_uri', 'flat_tuple', 'noop_body',\n           'respond', 'is_full_page', 'Redirect', 'get_key', 'qp', 'def_hdrs', 'FastHTML', 'HostRoute', 'nested_name',\n           'serve', 'Client', 'RouteFuncs', 'APIRouter', 'cookie', 'reg_re_param', 'StaticNoCache', 'MiddlewareBase',\n           'FtResponse', 'unqid']\n\n# %% ../nbs/api/00_core.ipynb #23503b9e\nimport json,\u2026</TRUNCATED>"
}
```

</details>



🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Good news — FastHTML has a built-in mechanism for exactly this: the `body_wrap` parameter. Looking at the source, `fast_app` / `FastHTML` accepts `body_wrap`, which is a function that wraps the body content of every full-page response. It receives `(content, req)` and its return value becomes the body content.

So instead of calling `mk_shell` in every route, you could do something like:

```python
def body_wrap(content, req):
    auth = req.scope.get('auth', None)
    return mk_nav(auth), Main(content, id='main')
```

Then pass it when creating your app:

```python
app, rt = fast_app(body_wrap=body_wrap, ...)
```

Your routes would then just return their content (plus `Title(...)` if needed — head-level tags like `Title` are automatically extracted and placed in `<head>`). So a route becomes:

```python
@rt
def index(sess):
    return Title('COBALT'), H1('Welcome')
```

This is cleaner than a decorator because it's built into FastHTML's rendering pipeline — it only applies to full-page responses (not HTMX partials), which is usually exactly what you want.

In [ ]:
#| export
@rt
def test(req, session):
    if not session.get('patient'): return Redirect(demographics.to())
    state = get_state(session)
    if len(state['reversals']) >= 6:
        threshold = sum(state['reversals'][-4:]) / 4
        return Div(Card(
            H2('Test Complete!'),
            P(f'Trials: {state["trial"]}'),
            P(f'Reversals: {len(state["reversals"])}'),
            P(f'Your TOJ Threshold: {threshold:.0f} ms', cls='text-xl font-bold'),
            Button('Start Over', hx_get=demographics, hx_target='#main', cls=ButtonT.primary), 
            cls='w-128'
        ), cls='min-h-screen flex items-center justify-center')

    side = random.choice(['left', 'right'])
    gap = state['gap_ms']

    return Div(Card(
        H2('Click Ordering Binaural Adaptive Lateralization Test (COBALT)'),
        P(f'Trial {state["trial"] + 1} — Gap: {state["gap_ms"]:.1f} ms — Reversals: {len(state["reversals"])}/6'),
        P('Put on headphones. Click Play, then indicate which ear heard the sound first.'),
        DivFullySpaced(Button('Play Sound', onclick=f'playTOJ("{side}", {gap})', cls=ButtonT.primary)),
        Div(
            Button('Left', hx_post=respond.to(answer='left', correct=side), hx_target='#result', cls=ButtonT.secondary),
            Button('Right', hx_post=respond.to(answer='right', correct=side), hx_target='#result', cls=ButtonT.secondary),
            cls='flex gap-4 justify-center mt-4'
        ),
        Div(id='result', cls='mt-4'),
        cls='w-128'
    ), cls='min-h-screen flex items-center justify-center')

In [ ]:
#| export
@rt
def respond(req, session, answer: str, correct: str):
    state = get_state(session)
    patient = session.get('patient', {})
    is_correct = answer == correct
    state['trial'] += 1
    state['results'].append(is_correct)
    trials.insert(Trial(initials=patient.get('initials'), birthday=patient.get('birthday'),
        sex=patient.get('sex'), test_date=str(date.today()), trial=state['trial'],
        gap_ms=state['gap_ms'], answer=answer, correct=correct))

    new_direction = None
    step = 0.8
    if is_correct:
        state['correct_streak'] += 1
        if state['correct_streak'] >= 2:
            new_direction = 'down'
            state['gap_ms'] = state['gap_ms'] * step
            state['correct_streak'] = 0
    else:
        state['correct_streak'] = 0
        new_direction = 'up'
        state['gap_ms'] = state['gap_ms'] / step

    if new_direction and state['last_direction'] and new_direction != state['last_direction']: state['reversals'].append(state['gap_ms'])
    if new_direction: state['last_direction'] = new_direction

    icon = "✓" if is_correct else "✗"
    return Div(
        P(f"{icon} {'Correct!' if is_correct else 'Incorrect'} (was {correct})"),
        Button("Next Trial", href=test, cls=ButtonT.primary),
    )

In [ ]:
#| export
@rt
def reset(session):
    session['state'] = mk_state()
    return Redirect(test)

In [ ]:
#| export
@rt
def index(sess): return mk_shell(sess, Container(render_md(Path('README.md').read_text())))

In [ ]:
#| export
db = database('cobalt.db')
class Trial: id:int; initials:str; birthday:str; sex:str; test_date:str; trial:int; gap_ms:float; answer:str; correct:str
trials = db.create(Trial, pk='id', transform=True)

In [ ]:
#| export
@rt
def demographics(req, session):
    return Div(Card(
        H3('Patient Information'),
        Form(LabelInput('Initials', id='initials', name='initials'),
             LabelInput('Birthday', id='birthday', name='birthday', type='date'),
             Label('Sex'),
             Select(Option('Male', value='M'), Option('Female', value='F'), Option('Other', value='O'), name='sex'),
             Button('Start Test', cls=ButtonT.primary, cls2='mt-4'),
             hx_post=save_demographics, hx_target='#main'),
        cls='w-96'
    ), cls='min-h-screen flex items-center justify-center')

@rt
def save_demographics(req, session, initials:str, birthday:str, sex:str):
    session['patient'] = dict(initials=initials, birthday=birthday, sex=sex)
    session['state'] = mk_state()
    return test(req, session)

In [ ]:
#| export
def patient_thresholds(sex=None, date_from=None, date_to=None):
    q = "SELECT initials,birthday,sex,test_date,gap_ms FROM trial t1 WHERE trial=(SELECT MAX(trial) FROM trial t2 WHERE t2.initials=t1.initials AND t2.test_date=t1.test_date)"
    conds,params = [],[]
    if sex: conds.append("sex=?"); params.append(sex)
    if date_from: conds.append("test_date>=?"); params.append(date_from)
    if date_to: conds.append("test_date<=?"); params.append(date_to)
    if conds: q = f"SELECT * FROM ({q}) WHERE " + " AND ".join(conds)
    return list(db.execute(q, params))

def mk_histogram(rows):
    vals = [r[4] for r in rows]
    if not vals: return None
    fig,ax = plt.subplots(figsize=(8,4))
    ax.hist(vals, bins=100, color='steelblue', edgecolor='white')
    ax.set(xlabel='Gap (ms)', ylabel='Count', title='TOJ Thresholds')
    buf = io.BytesIO()
    fig.savefig(buf, format='png', bbox_inches='tight')
    plt.close(fig)
    return base64.b64encode(buf.getvalue()).decode()

def mk_slicers(sex='', date_from='', date_to=''):
    frm = Form(
        Select(Option('All sexes',value=''),Option('Male',value='M'),Option('Female',value='F'),Option('Other',value='O'), name='sex'),
        LabelInput('From', id='date_from', name='date_from', type='date', value=date_from),
        LabelInput('To', id='date_to', name='date_to', type='date', value=date_to),
        Button('Filter', cls=ButtonT.primary),
        hx_get=data_chart, hx_target='#chart', cls='flex gap-4 items-end mt-4')
    return Div(frm, A('Download', href=data_csv.to(), cls=ButtonT.primary), cls='flex gap-4 items-end mt-4')

@rt
def data_chart(req, session, sex:str='', date_from:str='', date_to:str=''):
    rows = patient_thresholds(sex or None, date_from or None, date_to or None)
    img = mk_histogram(rows)
    return Div(P(f'{len(rows)} patient sessions'), Img(src=f'data:image/png;base64,{img}') if img else P('No data', cls='text-center text-gray-400 py-8'), id='chart')

@rt
def data_csv(req, session):
    buf = io.StringIO()
    w = csv.writer(buf)
    cols = [col.name for col in trials.columns]
    w.writerow(cols)
    w.writerows([[getattr(r, c) for c in cols] for r in trials()])
    return Response(buf.getvalue(), media_type='text/csv', headers={'Content-Disposition':'attachment; filename="cobalt_data.csv"'})

@rt
def data(req, session):
    rows = trials()
    return htmx(req, Container(
        H3('Trial Data'),
        mk_slicers(),
        Div(id='chart', hx_get=data_chart, hx_trigger='load')))

In [ ]:
#| export
serve()